In [1]:
import os

from dotenv import load_dotenv

load_dotenv("../.env")
neon_conn_string = os.getenv("NEON_DB_URL")
langchain_neon_conn_string = neon_conn_string.replace("postgresql", "postgresql+psycopg")

gpt_model = "gpt-5-mini"

Get the enums from the database

In [2]:
import psycopg

enums = ["availability_status_type", "room_bed_type", "room_status_type", "room_type"]
enum_values = {}

with psycopg.connect(neon_conn_string) as neon_conn:
    with neon_conn.cursor() as cur:
        for enum in enums:
            cur.execute(f"SELECT enum_range(NULL::{enum})")
            enum_values[enum] = [val.strip('\'"{}') for val in cur.fetchall()[0][0].split(',')]
        cur.execute("SELECT DISTINCT unnest(basic_amenities) FROM rooms;")
        basic_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(additional_amenities) FROM rooms;")
        additional_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(view_type) FROM rooms;")
        view_types = [val[0] for val in cur.fetchall()]


print(f"enums = {enum_values}")
print(f"basic amenities = {basic_amenities}")
print(f"additional amenities = {additional_amenities}")
print(f"view types = {view_types}")

enums = {'availability_status_type': ['Booked', 'Available', 'Maintenance'], 'room_bed_type': ['Queen', 'Double Queen', 'King', 'Double King', 'King + Sofa Bed', 'King + Multiple Sofa Beds'], 'room_status_type': ['Available', 'Occupied', 'Maintenance'], 'room_type': ['Standard', 'Deluxe', 'Suite', 'Presidential Suite']}
basic amenities = ['Full Kitchen', 'Executive Office', 'Nespresso Machine', 'Premium Coffee Maker', 'High-Speed WiFi', 'Premium Bathrobes', 'Kitchenette', 'Bathrobes', "Butler's Pantry", 'Welcome Amenity', 'Full-Size Refrigerator', '55" Smart TV', 'Bluetooth Speaker', 'Professional Coffee Bar', 'Guest Bathroom', 'Air Conditioning', 'Living Room', 'Luxury Welcome Amenity', 'Bang & Olufsen Sound System', 'Ultra-High-Speed WiFi', 'Work Desk', 'Multiple 75" Smart TVs', 'In-Room Safe', 'Smart TV', 'Living Room Area', 'Personalized Stationery', 'Hair Dryer', 'Walk-in Closet', 'Multiple Bathrooms', 'Dining Area', 'Bose Sound System', '65" Smart TV', 'Slippers', 'Evening Turndo

Write out the database schema and setup the connection for the LLM.

In [3]:
from langchain_community.utilities import SQLDatabase

schema_description = {
    "rooms": (f"""
            CREATE TABLE rooms (
                room_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_number INT NOT NULL,
                floor INT NOT NULL,
                type room_type, -- Enum with options {enum_values['room_type']}
                square_feet INT,
                basic_amenities TEXT[],  -- Options are {basic_amenities}
                additional_amenities TEXT[], -- Options are {additional_amenities}
                max_occupancy INT,
                bed_type room_bed_type,  -- Enum with options {enum_values['room_bed_type']}
                view_type TEXT[],  -- Options are {view_types}
                accessibility BOOLEAN,  -- Whether handicapped accessible
                status room_status_type, -- Enum with options {enum_values['room_status_type']}, do not return
                last_renovation DATE, -- Do not provide unless asked for
                base_rate NUMERIC(10, 2),  -- Do not provide unless asked for
                max_rate NUMERIC(10, 2)  -- Do not provide unless asked for
                -- The table lists details about all the rooms in the hotel.
            );
    """),
    "room_availability": (f"""
            CREATE TABLE room_availability (
                id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_id INT NOT NULL, -- Do not return
                room_number INT NOT NULL,
                date DATE NOT NULL,
                status availability_status_type,  -- Enum with options {enum_values['availability_status_type']}
                price NUMERIC(8,2),
                max_occupancy INT,
                FOREIGN KEY (room_id) REFERENCES rooms(room_id)
                -- The table lists the room availability by date and the corresponding rate
            );
    """),
}

db = SQLDatabase.from_uri(database_uri=langchain_neon_conn_string, include_tables=schema_description.keys(), custom_table_info=schema_description)

Setup the LangChain database toolkit

In [4]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI

# 1. Initialize your Language Model (LLM)
llm = ChatOpenAI(model=gpt_model, temperature=0)

# 2. Initialize the LangChain SQL Toolkit
# This toolkit will use the 'db' object with your custom schema info.
sql_toolkit = SQLDatabaseToolkit(db=db, llm=llm)

Setup the system prompt

In [5]:
dialect="PostgreSQL"
top_k=4

system_prompt = f"""system:
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.
Unless the user requests otherwise, limit your query to at most {top_k} results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.

You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
The ONLY exception is chaging a room from "available" to "booked" on specific dates when
making a reservation (booking a room) or from "booked" to "available" when canceling a
reservation. Only make a reservation (book a room) or cancel a reservation when the user
explicitly says to do so, only when the user has specified a room number, and only when
there is not ambiguity about what the user means. Then notify the user that you have
done so, along with the check-in and check-out dates and the total price of the booking.

To start you should ALWAYS look at the tables in the database to see what you can query.

Do NOT skip this step.

Then you should query the schema of the most relevant tables.

If, after querying the schema, anything about the user's query is unclear, ask for
clarification and stop.

If a date is mentioned without a year, assume that the year is 2025.

If asked for multiple consecutive nights, find available rooms by using "GROUP BY" on
the room number and counting the nights using "HAVING COUNT(*)."

Note that, when interacting with the user, a date range specifies from the date of
check-in to the date of check-out. So, January 1-3, 2025 would only be a stay of two
nights, and you would only query the database for January 1-2, 2025. To save the user
some confusion, always state the number of nights when you state a date range.

If you receive an empty result from an SQL query, that is acceptable. Simply use that
result and state that you couldn't find any relevant rooms. Do not try more than two
reformulations of the query.

Do not mention to the user that you are performing actions on a database. You can,
however, mention that you are performing a search.

If the user asks you anything unrelated to searching the database, booking a room, or
canceling a reservation, politely refuse.

After providing an answer to the user's question, booking a room, or canceling a
reservation, only offer to perform actions that involve querying the database or booking
a room. Do not offer to do anything else. You cannot provide a confirmation number or
send a confirmation email.
"""

print(system_prompt)

system:
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct PostgreSQL query to run, then look at the results of the query and return the answer.
Unless the user requests otherwise, limit your query to at most 4 results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.

You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
The ONLY exception is chaging a room from "available" to "booked" on specific dates when
making a reservation (boo

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=sql_toolkit.get_tools(),
    system_prompt=system_prompt,
)

In [8]:
prompt = 'Show me rooms with a view of the ocean and a 55" TV'
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

I found these rooms that have an ocean view and a 55" Smart TV:

- Room 1102 — Floor 11, Deluxe (King)
  - Sleeps: up to 3
  - Size: 465 sq ft
  - Views: Ocean View, Pool View
  - Basic amenities: Air Conditioning; 55" Smart TV; Nespresso Machine; Mini Fridge; Hair Dryer; In‑Room Safe; Work Desk; High‑Speed WiFi; Bluetooth Speaker; Microwave; Premium Bathrobes; Designer Slippers; Evening Turndown Service
  - Additional amenities: Soaking Tub; Lounge Access; Balcony
  - Accessibility: No

- Room 1104 — Floor 11, Deluxe (King)
  - Sleeps: up to 3
  - Size: 503 sq ft
  - Views: Ocean View, Pool View
  - Basic amenities: Air Conditioning; 55" Smart TV; Nespresso Machine; Mini Fridge; Hair Dryer; In‑Room Safe; Work Desk; High‑Speed WiFi; Bluetooth Speaker; Microwave; Premium Bathrobes; Designer Slippers; Evening Turndown Service
  - Additional amenities: Soaking Tub; Balcony
  - Accessibility: No

- Room 1106 — Floor 11, Deluxe (Double Queen)
  - Sleeps: up to 3
  - Size: 546 sq ft
  - View

In [ ]:
prompt = "How much to book the penthouse suite for the week of January 5th?"
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

{'messages': [HumanMessage(content='How much to book the penthouse suite for the week of January 5th?', additional_kwargs={}, response_metadata={}, id='78d029d1-e97d-4754-a6e3-15fccf7fa91b'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 1076, 'total_tokens': 1102, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Cm5LLaMm1hkVu6xEG37tHbaXp1CmS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b148c-99ae-7781-b86b-60867d9c1a48-0', tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'call_Q48kLyBbUNewYAxC2WhLFZE8', 'type': 'tool_call'}], usage_metadata={'input_tokens': 

In [10]:
from pprint import pprint
pprint(result)

{'messages': [HumanMessage(content='How much to book the penthouse suite for the week of January 5th?', additional_kwargs={}, response_metadata={}, id='78d029d1-e97d-4754-a6e3-15fccf7fa91b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 1076, 'total_tokens': 1102, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Cm5LLaMm1hkVu6xEG37tHbaXp1CmS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b148c-99ae-7781-b86b-60867d9c1a48-0', tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'call_Q48kLyBbUNewYAxC2WhLFZE8', 'type': 'tool_call'}], usage_metadata={'i